---
title: Reproducing the solar cloud radiative effect with libRadtran
label: libradtran-cre-example
---
# libRadtran &mdash; the solar cloud radiative effect over Arctic sea ice

Author of this notebook:
 - *Your name*, [Leipzig Institute for Meteorology](https://www.physgeo.uni-leipzig.de/institut-fuer-meteorologie), Leipzig University, Germany, *email*

This notebook is licensed under the [Creative Commons Attribution 4.0 International](http://creativecommons.org/licenses/by/4.0/ "CC-BY-4.0")

## Dataset description

**Title:** Aircraft measurements of broadband irradiance during the ACLOUD campaign in 2017

**Authors:** Stapf, Johannes; Ehrlich, André; Jäkel, Evelyn; Wendisch, Manfred

**Description:** Solar and terrestrial broadband irradiance and nadir brightness temperature measured by upward- and downward-looking pyranometers and pyrgeometers aboard the *Polar 5* aircraft during the ACLOUD campaign [@wendisch2019] north-west of Svalbard in May/June 2017. One netCDF file is provided per research flight.

**Year:** 2019

**Institute:** Leipzig University

**DOI:** [10.1594/PANGAEA.900442](https://doi.org/10.1594/PANGAEA.900442) [@stapf2019data]

**License:** [Creative Commons Attribution 4.0 International](http://creativecommons.org/licenses/by/4.0/)

## Contents of this notebook

This notebook is a compact showcase of [`pyRadtran`](https://github.com/FranzFlink/pyRadtran), a thin `xarray` wrapper around the radiative transfer library **libRadtran**. It reproduces the key idea behind Fig. 7 of @becker2023: the **solar cloud radiative effect** (CRE$_\mathrm{sol}$) as a function of the underlying **surface albedo** for the ACLOUD campaign. To do so we

1. download the airborne broadband irradiance observations from PANGAEA,
2. run a **clear-sky** libRadtran simulation along the flight track with `pyRadtran`, and
3. combine measurement and simulation to obtain CRE$_\mathrm{sol}$ and visualise it as a two-dimensional probability density.


## A short introduction to libRadtran and pyRadtran

**libRadtran** is a widely used, freely available *library for radiative transfer* calculations [@mayer2005; @emde2016]. Its main tool, `uvspec`, solves the radiative transfer equation through a user-defined atmosphere and returns spectral or broadband irradiances and radiances. The radiative transfer equation is solved by one of several solvers; the default for irradiances is **DISORT**, the discrete-ordinate method of @stamnes1988. A simulation is configured through a plain-text input file that specifies, e.g., the `source` (solar or thermal), the atmospheric profile, trace-gas amounts, the surface `albedo`, the solar zenith angle and the wavelength range.

The **cloud radiative effect** (CRE) is the difference between the net irradiance of the *real* (cloudy) atmosphere and that of an otherwise identical *cloud-free* atmosphere. The cloudy term is measured by the aircraft; the cloud-free reference cannot be observed and is instead **simulated with libRadtran** &mdash; exactly the approach taken by @becker2023.

[`pyRadtran`](https://github.com/FranzFlink/pyRadtran) makes this workflow accessible from Python: it registers a `.pyradtran` accessor on `xarray.Dataset` objects, builds the `uvspec` input files for every point of a space–time grid, runs them (optionally in parallel) and returns the results as an `xarray.Dataset`.

```{note}
Running the simulation cells below requires a working **libRadtran** installation and the `pyradtran` package. libRadtran does not run inside the browser-based book build, so the simulation cells are shipped without stored output &mdash; execute them on a machine where `uvspec` is available.
```


## Import relevant modules

Besides the usual scientific stack (`numpy`, `xarray`, `matplotlib`) we use [`pangaeapy`](https://pypi.org/project/pangaeapy/) to query the PANGAEA data publication and `pyradtran` for the radiative transfer simulation.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import pangaeapy as pgp   # query PANGAEA datasets
import pyradtran          # registers the .pyradtran xarray accessor

## Download the broadband irradiance data from PANGAEA

The parent dataset [10.1594/PANGAEA.900442](https://doi.org/10.1594/PANGAEA.900442) is a *collection* of netCDF files, one per research flight. We first list the available files and then download a single flight (here research flight RF14 on 8 June 2017, which sampled both sea-ice and open-ocean legs).

In [ ]:
# List the files contained in the PANGAEA collection
catalogue = pgp.PanDataSet(900442, enable_cache=True)
print(catalogue.title)
catalogue.data.head()

In [ ]:
# Download one research flight (adapt the date to pick another flight)
flight_url = (
    "https://hs.pangaea.de/Projects/AC3/ACLOUD_irradiance/"
    "ACLOUD_20170608_Airborne_Broadband_Irradiance.nc"
)
local_file = Path(flight_url.split("/")[-1])
if not local_file.exists():
    urllib.request.urlretrieve(flight_url, local_file)

obs = xr.open_dataset(local_file)
obs

## Pre-process the observations

From the up- and downward broadband solar irradiances we obtain the **measured net solar irradiance** and the **surface albedo**.

```{warning}
The variable names below (`F_down_solar`, `F_up_solar`, `lat`, `lon`) are placeholders &mdash; inspect the `obs` dataset printed above and rename them to match the file.
```

In [ ]:
# ---- adjust the variable names to match the dataset printed above ----
F_dw = obs["F_down_solar"]    # measured downward broadband solar irradiance (W m-2)
F_up = obs["F_up_solar"]      # measured upward   broadband solar irradiance (W m-2)

# measured broadband surface albedo and net (downward-positive) solar irradiance
albedo = (F_up / F_dw).clip(0.0, 1.0)
F_net_meas = F_dw - F_up

## Clear-sky simulation with pyRadtran

We build an `xarray.Dataset` carrying the flight track (`time`, `latitude`, `longitude`) and hand it to the `.pyradtran` accessor. The configuration file [`config/clearsky_solar.yaml`](./config/clearsky_solar.yaml) selects a **clear-sky, broadband solar** simulation (DISORT solver, 200–3600 nm, subarctic-summer atmosphere); the solar zenith angle is computed automatically from time and position.

To keep the demonstration short the track is sub-sampled &mdash; remove the `stride` to simulate every measurement point.

In [ ]:
stride = 60   # sub-sample the high-rate track for a quick demo

track = xr.Dataset(
    coords={
        "time": obs["time"][::stride],
        "latitude": ("time", np.asarray(obs["lat"][::stride])),
        "longitude": ("time", np.asarray(obs["lon"][::stride])),
    }
)

# Run libRadtran along the track (requires uvspec + pyradtran)
sim = track.pyradtran.run(
    config_path=Path("config/clearsky_solar.yaml"),
    return_dataset=True,
    save_to_file=True,
    show_progress=True,
)

# clear-sky global (direct + diffuse) downward solar irradiance at flight level
F_dw_clear = sim["eglo"].squeeze()
F_dw_clear

## The solar cloud radiative effect

Following @becker2023, the solar cloud radiative effect is the difference between the measured and the simulated clear-sky **net** solar irradiance,

$$\mathrm{CRE_{sol}} = \left(F^{\downarrow} - F^{\uparrow}\right)_\mathrm{meas} - \left(F^{\downarrow} - F^{\uparrow}\right)_\mathrm{clear}.$$

The clear-sky downward irradiance is almost independent of the surface albedo, whereas the clear-sky *upward* irradiance scales with it. We therefore reconstruct the clear-sky net flux from the measured albedo as $F_\mathrm{net,clear} = (1-\alpha)\,F^{\downarrow}_\mathrm{clear}$, after interpolating the simulation back onto the full measurement grid.

In [ ]:
# interpolate the (sub-sampled) clear-sky simulation onto the measurement grid
F_dw_clear_full = F_dw_clear.interp(time=obs["time"])

# clear-sky net solar irradiance and the solar cloud radiative effect
F_net_clear = (1.0 - albedo) * F_dw_clear_full
cre_sol = F_net_meas - F_net_clear

## Reproducing Fig. 7: CRE$_\mathrm{sol}$ versus surface albedo

Finally we visualise the relationship between the solar cloud radiative effect and the surface albedo as a two-dimensional probability density &mdash; the core panel of Fig. 7 in @becker2023. The cloud-free ocean, cloud-free ice and cloudy regimes appear as distinct modes; cooling (negative CRE$_\mathrm{sol}$) dominates over the dark ocean while the effect weakens over the bright sea ice.

In [ ]:
a = np.asarray(albedo).ravel()
c = np.asarray(cre_sol).ravel()
m = np.isfinite(a) & np.isfinite(c)

fig, ax = plt.subplots(figsize=(7, 5))
hb = ax.hist2d(
    a[m], c[m], bins=(40, 40), range=[[0, 1], [-200, 100]],
    cmap="cividis", density=True,
)
fig.colorbar(hb[3], ax=ax, label="probability density")
ax.axhline(0.0, color="k", lw=0.8, ls="--")
ax.set_xlabel("surface albedo")
ax.set_ylabel(r"solar cloud radiative effect CRE$_\mathrm{sol}$ (W m$^{-2}$)")
ax.set_title("ACLOUD — solar cloud radiative effect vs. surface albedo")
plt.show()